In [1]:
!pip install transformers datasets evaluate captum scikit-learn pandas -q


# Adversarial Evaluation
Applying perturbations and checking Attack Success Rate (ASR) across all 5 architectures.


In [2]:
import pandas as pd
import sys
import os
sys.path.append(os.path.abspath('../src'))
from models.lr_tfidf import LRTfidfModel
from models.roberta import RobertaModel
from models.bert import BertModel
from models.albert import AlbertModel
from evaluation import compute_attack_success_rate


In [3]:
adv_df = pd.read_pickle('../data/processed/adv_test.pkl')
orig_true = adv_df['label'].tolist()


In [4]:
models = {
    'lr_char': LRTfidfModel(model_dir='../models/lr_char', analyzer='char_wb'),
    'lr_word': LRTfidfModel(model_dir='../models/lr_word', analyzer='word'),
    'roberta': RobertaModel(model_dir='../models/roberta'),
    'bert': BertModel(model_dir='../models/bert'),
    'albert': AlbertModel(model_dir='../models/albert')
}


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


In [5]:
for name, model in models.items():
    model.load()
    if 'lr' in name:
        orig_preds = model.predict(adv_df['original_text'].tolist())
        adv_preds = model.predict(adv_df['perturbed_text'].tolist())
    else:
        orig_preds, _ = model.predict(adv_df['original_text'].tolist(), batch_size=16)
        adv_preds, _ = model.predict(adv_df['perturbed_text'].tolist(), batch_size=16)
    asr = compute_attack_success_rate(orig_true, orig_preds, adv_preds)
    print(f'Overall Attack Success Rate ({name.upper()}): {asr:.2%}')


Model loaded.


Overall Attack Success Rate (LR_CHAR): 16.02%


Model loaded.


Overall Attack Success Rate (LR_WORD): 14.95%


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Overall Attack Success Rate (ROBERTA): 34.91%


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Overall Attack Success Rate (BERT): 47.21%


Loading weights:   0%|          | 0/27 [00:00<?, ?it/s]

Overall Attack Success Rate (ALBERT): 42.26%
